# LangChain L12 — Level 11 — LangGraph workflows and persistence
Some processes should not be left to a model's judgement at every step. A support ticket at
Meridian must follow a fixed shape: classify, gather evidence in the right system, draft, get
human approval, send. That is a **workflow**, and `create_agent()`'s free-form loop is the
wrong tool for it. LangGraph is the layer underneath: you declare the graph yourself.

```text
START -> classify -+-> faq (policy search) ---+-> draft -> approve (interrupt) -> send -> END
                   +-> billing (order lookup) +
```

- **State** is a typed dictionary that flows through the graph.
- **Nodes** are Python functions that read state and return updates.
- **Edges** connect nodes; **conditional edges** choose the next node from state.
- A **checkpointer** makes the graph resumable; `interrupt()` pauses it inside a node.

`create_agent()` is itself a LangGraph graph with a *model* node and a *tools* node. Once you
can build this section's graph, you can build any agent shape, and you can mix both: an agent
can be one node of a larger workflow.

### Step 0 — Graph vocabulary with toy examples (no model involved)

Before the real workflow, three toy graphs make the vocabulary concrete. None of them calls a
model: LangGraph is just a way to run Python functions in a declared order.

```text
STATE   a dictionary that flows through the graph; nodes read it and return partial updates
NODE    a Python function  state -> {changed keys}
EDGE    "after node A, run node B"
CONDITIONAL EDGE   "after node A, call a routing function on the state; it names the next node"
START / END        where a run enters and leaves
```

In [ ]:
from typing import TypedDict                           # Python standard library
from langgraph.graph import StateGraph, START, END     # LangGraph: the graph builder and its two fixed endpoints

# --- Toy 1: two nodes in a row. State is a dict with one number in it. ------------------------
class Counter(TypedDict):                              # ours: the state schema
    value: int

def add_ten(state: Counter):                           # ours: a node = function(state) -> partial update
    return {"value": state["value"] + 10}

def double(state: Counter):                            # ours
    return {"value": state["value"] * 2}

toy = StateGraph(Counter)                              # LangGraph: start declaring a graph over this state
toy.add_node("add_ten", add_ten)                       # LangGraph: register nodes by name
toy.add_node("double", double)
toy.add_edge(START, "add_ten")                         # LangGraph: edges = order of execution
toy.add_edge("add_ten", "double")
toy.add_edge("double", END)
toy_graph = toy.compile()                              # LangGraph: turn the declaration into something runnable
print("toy 1 :", toy_graph.invoke({"value": 1}), "   (1 + 10) * 2")   # LangGraph: invoke() runs START -> ... -> END

# --- Toy 2: a conditional edge chooses the path from the state. ------------------------------
def classify_number(state: Counter):                   # ours: a routing function returns the NAME of the next node
    return "double" if state["value"] % 2 == 0 else "add_ten"

branchy = StateGraph(Counter)
branchy.add_node("add_ten", add_ten)
branchy.add_node("double", double)
branchy.add_conditional_edges(START, classify_number, {"double": "double", "add_ten": "add_ten"})   # LangGraph
branchy.add_edge("add_ten", END)
branchy.add_edge("double", END)
branchy_graph = branchy.compile()
print("toy 2 :", branchy_graph.invoke({"value": 4}), "(even -> double)  ", branchy_graph.invoke({"value": 5}), "(odd -> add_ten)")

# --- Toy 3: a loop. An edge back to an earlier node repeats until a condition says stop. -------
def keep_going(state: Counter):                        # ours: loop condition
    return "add_ten" if state["value"] < 50 else END

loopy = StateGraph(Counter)
loopy.add_node("add_ten", add_ten)
loopy.add_edge(START, "add_ten")
loopy.add_conditional_edges("add_ten", keep_going, {"add_ten": "add_ten", END: END})   # LangGraph: edge back to itself
loopy_graph = loopy.compile()
print("toy 3 :", loopy_graph.invoke({"value": 5}), "   5 -> 15 -> 25 -> 35 -> 45 -> 55, then stop")
print("\nThe agent loop of L3 is toy 3 with 'model' and 'tools' as the nodes and 'any tool calls?' as the condition.")

### Step 1 — State and nodes

Now the real workflow. Each node does one job and returns only the keys it changes. Model calls
appear where they add value (classification, drafting); deterministic work (order lookup) is plain Python.

In [ ]:
from langgraph.types import interrupt                  # LangGraph: pause a run inside a node

class RouteDecision(BaseModel):                        # ours, on Pydantic
    """Which desk should handle the ticket."""
    category: Literal["faq", "billing"] = Field(description="billing for orders, charges and refunds; faq for policy questions.")

def structured(model, schema):                         # ours: a one-line wrapper
    """model.with_structured_output for the course model (function calling is the most portable method on OpenRouter)."""
    return model.with_structured_output(schema, method="function_calling") if LIVE else model.with_structured_output(schema)   # LangChain

class TicketState(TypedDict, total=False):             # ours: the workflow state
    question: str
    category: str
    evidence: str
    draft: str
    approved: bool
    final: str

def classify(state: TicketState):
    decision = structured(model, RouteDecision).invoke([HumanMessage(state["question"])])
    return {"category": decision.category}

def faq(state: TicketState):
    return {"evidence": search_policies.invoke({"query": state["question"]})}

def billing(state: TicketState):
    order_id = re.search(r"O\d{4}", state["question"])
    return {"evidence": get_order.invoke({"order_id": order_id.group(0)}) if order_id else "no order id in the question"}

def draft(state: TicketState):
    reply = model.invoke([SystemMessage("Draft a short customer reply from the evidence. Be factual.\n\nEvidence:\n" + state["evidence"]), HumanMessage(state["question"])])
    return {"draft": text_of(reply)}

def approve(state: TicketState):
    decision = interrupt({"draft": state["draft"], "question": "Send this reply to the customer?"})   # LangGraph: pauses here; resumes with the human's value
    return {"approved": bool(decision)}

def send(state: TicketState):
    return {"final": state["draft"] if state["approved"] else "Reply withheld by reviewer."}

print("nodes defined:", [f.__name__ for f in (classify, faq, billing, draft, approve, send)])

### Step 2 — Wire the graph, compile with a checkpointer, run to the pause

In [ ]:
builder = StateGraph(TicketState)                      # LangGraph
for node in (classify, faq, billing, draft, approve, send):
    builder.add_node(node.__name__, node)               # LangGraph: our functions become nodes
builder.add_edge(START, "classify")                    # LangGraph
builder.add_conditional_edges("classify", lambda state: state["category"], {"faq": "faq", "billing": "billing"})   # LangGraph: route on state
builder.add_edge("faq", "draft")
builder.add_edge("billing", "draft")
builder.add_edge("draft", "approve")
builder.add_edge("approve", "send")
builder.add_edge("send", END)
ticket_graph = builder.compile(checkpointer=InMemorySaver())   # LangGraph: compile with persistence

print(ticket_graph.get_graph().draw_mermaid())        # LangGraph: the same picture as a Mermaid diagram

run = {"configurable": {"thread_id": "wf-1"}}
paused = ticket_graph.invoke({"question": "I was charged twice for order O1002. What happens now?"}, run)
print("category :", paused["category"])
print("evidence :", paused["evidence"][:90])
print("paused at:", ticket_graph.get_state(run).next, "| asks:", paused["__interrupt__"][0].value["question"])

### Step 3 — Resume, and see durability

The reviewer approves; the graph continues from the `approve` node, not from the start.
`get_state_history()` lists every checkpoint: this is what makes crash recovery and
"time travel" debugging possible, and why long-running agents are built on persistence.

In [ ]:
finished = ticket_graph.invoke(Command(resume=True), run)   # LangGraph: resume the paused thread with the human's answer
print("final    :", finished["final"][:140])

history = list(ticket_graph.get_state_history(run))     # LangGraph: every checkpoint of this thread
print("\ncheckpoints recorded:", len(history))
for snap in reversed(history):
    print("  next =", snap.next or ("END",), "| keys so far:", sorted(k for k in snap.values if k != "question"))

### Recap

- **Problem seen:** a fixed business process was being left to a free-form agent loop.
- **Layer added:** an explicit LangGraph: typed state, nodes, conditional edges, `interrupt()`, checkpoints and state history.
- **Evidence:** the ticket paused at approval, resumed from that exact node, and every step was recorded as a checkpoint.